Här skapar jag en komplett RAG-chattbot med en tydlig prompt. Prompten styr modellen till att ge fullständiga svar med förklaringar och rätt terminologi och bara svara utifrån den kontext som hämtats från vår VectorStore

Flödet:
1. Frågan omvandlas först med hjälp av EmbeddingService.
2. VectorStore jämför frågans embedding med embeddings från de sparade dokumenten. Med cosine similarity så hämtas de 5 mest relevanta chunksen.
3. De chunks som hittas sätts ihop med kontext i prompten och skickas sedan till den valda LLM:en som genererar ett svar.

In [3]:

import os
from dotenv import load_dotenv
from google import genai
from google.genai import types
from embedding_service import EmbeddingService
from vector_store import VectorStore


def build_prompt(query, results):
    """
    Bygger en enkel prompt med hämtad kontext.
    """
    context_parts = []

    for i, row in enumerate(results, start=1):
        text = row["text"].strip()
        context_parts.append(f"Källa {i}:\n{text}")

    context = "\n\n".join(context_parts)

    prompt = f"""
Du är en AI-assistent som hjälper skeppare och värdar på More Sailing att förstå kursmaterialet på ett korrekt och professionellt sätt.

Ditt mål är att:
- Ge korrekta och fullständiga svar baserade på kursmaterialet
- Hjälpa användaren att förstå detaljer och sammanhang
- Fungera som en pålitlig kunskapskälla

Dina svar ska:
- Vara tydliga och välstrukturerade
- Innehålla tillräckligt med detaljer för att ge en korrekt förståelse
- Använda korrekt terminologi från kursmaterialet

Regler:
- Använd ENDAST informationen i kontexten nedan
- Hitta inte på information
- Om svaret inte finns i kontexten: säg "Jag vet inte baserat på kursmaterialet"

Om möjligt:
- Förklara begrepp tydligt
- Ge sammanhang så att användaren förstår varför något görs
- Knyt till praktiska situationer om det framgår i materialet

När du svarar:
- Prioritera konkreta instruktioner och regler från kursmaterialet
- Ta med viktiga åtgärder (t.ex. att informera ansvarig)
- Fokusera inte enbart på tekniska detaljer utan även beteende och ansvar

Svara i följande struktur:
1. Kort svar
2. Förklaring
3. Eventuella viktiga detaljer eller undantag

KONTEXT:
{context}

FRÅGA:
{query}
"""

    return prompt.strip()


if __name__ == "__main__":
    load_dotenv()

    client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

    # Laddar embedding-service för query embeddings
    service = EmbeddingService(client, types)

    # Laddar färdiga embeddings från fil
    store = VectorStore("embeddings_v2.parquet")

    print("*** More Sailing RAG Chat ***")
    print("Skriv q för att avsluta.\n")

    while True:
        query = input("Du: ").strip()

        if query.lower() == "q":
            print("Avslutar chatten.")
            break

        if not query:
            continue

        try:
            # 1. Embedding av fråga
            query_embedding = service.embed(query)

            # 2. Hämta mest relevanta chunks
            results = store.search(query_embedding, k=5)

            # 3. Bygg prompt
            prompt = build_prompt(query, results)

            # 4. Skicka till modell
            response = client.models.generate_content(
                model="gemini-3.1-flash-lite-preview",
                contents=prompt
            )
            # 5. Skicka svaret
            print(f"\nFråga: {query}")
            print("\nBot:", response.text)
            print("\n" + "-" * 80 + "\n")

        except Exception as e:
            print(f"\nFel: {e}\n")

*** More Sailing RAG Chat ***
Skriv q för att avsluta.


Fråga: Hur ser en vanlig chartervecka ut?

Bot: En vanlig chartervecka hos More Sailing präglas av ett strukturerat dagsschema för både besättning och gäster, samt specifika rutiner för underhåll och kommunikation.

### 1. Kort svar
En vanlig vecka följer en daglig rutin med frukost, segling, lunchstopp och ankomst till nattens hamn eller vik. Veckan inleds och avslutas med bytesdagar (oftast lördagar) där förberedelser och teknikgenomgångar är centrala. Som personal arbetar skeppare och värd tätt ihop med både praktiska uppgifter och kontinuerlig avstämning.

### 2. Förklaring
*   **Daglig rutin:**
    *   **08.00:** Frukost serveras och ett morgonmöte hålls där dagen planeras. Under tiden förbereder skepparen båten för avgång (köper bröd, betalar hamnavgift).
    *   **09.00:** Avfärd.
    *   **Lunchtid (ca 12–13):** Stopp i en vik för bad och mat.
    *   **Sen eftermiddag (ca 15–18):** Ankomst till nattens destination, tillä